# 03 Embeddings (Colab)

Embeds every course and student profile with Sentence-BERT (`all-MiniLM-L6-v2`) and OpenAI `text-embedding-3-small` (512 dimensions). TF-IDF doesn't need this step.

Needs `OPENAI_API_KEY` in Colab Secrets. Takes 5 to 10 minutes and costs about $0.02.

In [1]:
# Step 0: setup (works in Colab and on a laptop)
import os, sys
from pathlib import Path

# Colab: where the project folder sits in Google Drive
DRIVE_PROJECT = "/content/drive/MyDrive/LearnPath"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path(DRIVE_PROJECT)
else:
    PROJECT_DIR = Path.cwd().parent   # running from the notebooks/ folder

assert (PROJECT_DIR / "src" / "learnpath").exists(), f"Project folder not found: {PROJECT_DIR}"
os.environ["LEARNPATH_DIR"] = str(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / "src"))
print("Project folder:", PROJECT_DIR)

Mounted at /content/drive
Project folder: /content/drive/MyDrive/LearnPath


In [2]:
# Step 1: install
!pip -q install sentence-transformers openai

In [3]:
# Step 2: texts to embed
import numpy as np
import pandas as pd
from learnpath import catalog, config, retrieval

courses = catalog.load_courses()
students = pd.read_csv(config.STUDENTS)
cold = pd.read_csv(config.COLD_START)
people = pd.concat([students, cold], ignore_index=True)

course_texts = courses["doc"].tolist()
profile_texts = [retrieval.profile_text(r) for r in people.to_dict("records")]
print(len(course_texts), "course documents,", len(profile_texts), "student profiles")
print("Example profile:", profile_texts[0])

3422 course documents, 2200 student profiles
Example profile: Interested in leadership and management; environmental science and sustainability. Goal: career switch. Level: Beginner. Prefers reading.


In [4]:
# Step 3: Sentence-BERT
from sentence_transformers import SentenceTransformer

sbert = SentenceTransformer(config.SBERT_MODEL)
C_sbert = sbert.encode(course_texts, batch_size=64, show_progress_bar=False, normalize_embeddings=True)
P_sbert = sbert.encode(profile_texts, batch_size=64, show_progress_bar=False, normalize_embeddings=True)
retrieval.save_embeddings("sbert", courses["course_id"].tolist(), C_sbert,
                          people["student_id"].tolist(), P_sbert)
print("Saved Sentence-BERT vectors:", C_sbert.shape, P_sbert.shape)

Saved Sentence-BERT vectors: (3422, 384) (2200, 384)


In [5]:
# Step 4: one small test call to catch a wrong key or model name early
from learnpath.llm import make_client

client = make_client("openai")
test = client.embeddings.create(model=config.OPENAI_EMBED_MODEL, input=["hello"], dimensions=config.OPENAI_EMBED_DIM)
print("OK, vector length:", len(test.data[0].embedding))

OK, vector length: 512


In [6]:
# Step 5: OpenAI embeddings for everything
# uses OpenAI credit, about $0.02 in total
import time

def embed_all(texts, batch=100):
    out, tokens = [], 0
    for i in range(0, len(texts), batch):
        for attempt in range(3):
            try:
                r = client.embeddings.create(model=config.OPENAI_EMBED_MODEL,
                                             input=texts[i:i + batch],
                                             dimensions=config.OPENAI_EMBED_DIM)
                break
            except Exception as e:
                print("retrying after error:", e)
                time.sleep(2 ** attempt)
        out += [d.embedding for d in r.data]
        tokens += r.usage.total_tokens
        if (i // batch) % 10 == 0:
            print(f"{i + len(texts[i:i + batch])}/{len(texts)}")
    return np.array(out, dtype=np.float32), tokens

C_oa, t1 = embed_all(course_texts)
P_oa, t2 = embed_all(profile_texts)
retrieval.save_embeddings("openai", courses["course_id"].tolist(), C_oa,
                          people["student_id"].tolist(), P_oa)
print("Saved OpenAI vectors:", C_oa.shape, P_oa.shape)
print(f"Tokens used: {t1 + t2:,}  (about ${(t1 + t2) * 0.02 / 1e6:.3f})")

100/3422
1100/3422
2100/3422
3100/3422
100/2200
1100/2200
2100/2200
Saved OpenAI vectors: (3422, 512) (2200, 512)
Tokens used: 791,092  (about $0.016)


In [7]:
# Step 6: sanity check, neighbors of an ML course should also be about ML
from sklearn.preprocessing import normalize

probe = courses.index[courses["name"].str.contains("Machine Learning", case=False)][0]
print("Probe:", courses.loc[probe, "name"])
for name, M_ in [("Sentence-BERT", C_sbert), ("OpenAI", C_oa)]:
    Mn = normalize(M_)
    sims = Mn @ Mn[probe]
    top = np.argsort(-sims)[1:6]
    print(f"\n{name}:")
    for i in top:
        print(f"  {sims[i]:.2f}  {courses.loc[i, 'name']}")

Probe: Project Planning and Machine Learning

Sentence-BERT:
  0.74  Machine Learning With Big Data
  0.72  Introduction to Applied Machine Learning
  0.71  Optimizing Machine Learning Performance
  0.71  Big Data Applications: Machine Learning at Scale
  0.70  Structuring Machine Learning Projects

OpenAI:
  0.70  Modeling and Debugging Embedded Systems
  0.70  Advanced Machine Learning and Signal Processing
  0.70  Sensor Manufacturing and Process Control
  0.69  Optimizing Machine Learning Performance
  0.69  Introduction to Applied Machine Learning


Next: notebook 04.